# DICOM to BIDS Conversion — Single Subject (`heudiconv`)

This notebook walks through DICOM-to-BIDS conversion using `heudiconv` inside a Singularity container.

It is organized into two phases:

- **1. (New projects only):** Run `heudiconv` in heuristic/scan mode on a representative subject to generate a `dicominfo.tsv` describing all scan series. Use this to write your `heuristic.py` configuration file.
- **2. (All projects):** Run `heudiconv` in conversion mode using your `heuristic.py` to convert DICOMs to NIfTIs in a BIDS-compliant structure.

For batch conversion across many subjects, see `heudiconv_batch.ipynb`.

---

### References
- [heudiconv documentation](https://heudiconv.readthedocs.io/en/latest/)
- [Stanford BIDS tutorial (heudiconv)](http://reproducibility.stanford.edu/bids-tutorial-series-part-2a/)
- [BIDS specification](https://bids-specification.readthedocs.io/)

#### History
- 4/1/20 mbod — initial setup for MURI DICOMs
- 9/9/21 dcosme — separated into single-subject and loop notebooks
- Refactored for CNLab pipeline documentation

## 1. Imports and Setup

In [ ]:
import os
import pandas as pd

## 2. Set Paths and Variables

Edit the variables below for your project. All other cells derive from these.

> **Geoscan reference:** For the geoscan project, `project = 'geoscan_v2'`, subject IDs followed the format `GEO###`, and DICOMs were organized under `/fmriDataRaw/fmri_data_raw/geoscan/T2/{subject}_T2/` with a session label of `t2` or `t3`.

In [ ]:
# ── Project paths ─────────────────────────────────────────────────────────────
project       = 'your_project'           # Project folder name under /data00/projects/
project_dir   = f'/data00/projects/{project}'
bids_dir      = os.path.join(project_dir, 'data/bids_data')
heudiconv_dir = os.path.join(project_dir, 'scripts/BIDS/heudiconv')
heuristic     = os.path.join(heudiconv_dir, 'code/heuristic.py')

# ── Raw DICOM path ────────────────────────────────────────────────────────────
# This is the pattern heudiconv uses to find DICOMs. {subject} is filled in automatically.
# Adjust the subdirectory structure to match your project's DICOM organization.
#
# Geoscan example: /raw/geoscan/T2/{subject}_T2/*.dcm
raw_dicom_pattern = '/raw/your_project/{subject}/*/*.dcm'

# ── Singularity image ─────────────────────────────────────────────────────────
# Link to your heudiconv singularity image.
heudiconv_sif = '/data00/tools/singularity_images/heudiconv_0.8.0'

# ── Subject and session ───────────────────────────────────────────────────────
sub     = 'sub-001'    # Subject ID as it appears in the DICOM folder name
session = 't1'         # Session label (e.g. 't1', 't2', 'baseline')

# ── Verify directories exist ──────────────────────────────────────────────────
print(f"Project dir  : {project_dir}")
print(f"BIDS dir     : {bids_dir}")
print(f"heudiconv dir: {heudiconv_dir}")
print(f"Subject      : {sub}")
print(f"Session      : {session}")
print(f"\nBIDS dir exists    : {os.path.exists(bids_dir)}")
print(f"heuristic.py exists: {os.path.exists(heuristic)}")

---
## Phase 1: Scan Mode (New Projects Only)

**Skip this phase if your project already has a working `heuristic.py`.** Go directly to Phase 2.

This phase runs `heudiconv` with `-c none` (no conversion) on one representative subject to produce:
- `dicominfo.tsv` — a table describing every scan series found in the DICOMs
- `heuristic.py` — a template configuration file for you to fill in

Choose a subject that has a **complete and representative set of scans** (all tasks, all field maps, anatomicals).

In [ ]:
# Phase 1: Scan DICOMs and generate dicominfo.tsv
# Output goes to heudiconv/.heudiconv/{sub}/info/
#
# -f convertall : include all scan types
# -c none       : scan only, do not convert
# --overwrite   : overwrite previous scan output if it exists

!singularity run --cleanenv \
    -B {project_dir}:/base \
    -B /fmriDataRaw/fmri_data_raw:/raw \
    {heudiconv_sif} \
    -d {raw_dicom_pattern} \
    -o heudiconv/ \
    -f convertall \
    -s {sub} \
    -ss {session} \
    -c none \
    --overwrite

### Inspect the Phase 1 output

The output is stored in a hidden `.heudiconv` folder. The `dicominfo.tsv` is the key file — it describes every scan series found.

In [ ]:
# List the generated files
!ls -a heudiconv/.heudiconv/{sub}/info

In [ ]:
# Load dicominfo.tsv into a dataframe for inspection
dicominfo_path = f'heudiconv/.heudiconv/{sub}/info/dicominfo_ses-{session}.tsv'
scan_df = pd.read_csv(dicominfo_path, sep='\t')

# Show the fields most useful for writing heuristic.py
# series_id / series_description : how to identify scan types
# dim4                           : number of volumes (use to filter partial/aborted runs)
scan_df[['series_id', 'series_description', 'dim1', 'dim2', 'dim3', 'dim4']]

### Write `heuristic.py`

Using the `dicominfo.tsv` output above, edit your `heuristic.py` at:
```
{project}/scripts/BIDS/heudiconv/code/heuristic.py
```

The file needs two things:

**1. KEYS** — BIDS output path templates defined with `create_key()`. Each key corresponds to one scan type:
```python
def infotodict(seqinfo):
    # Anatomical
    t1w = create_key('sub-{subject}/ses-{session}/anat/sub-{subject}_ses-{session}_T1w')
    t2w = create_key('sub-{subject}/ses-{session}/anat/sub-{subject}_ses-{session}_T2w')

    # Functional runs — {item:01d} auto-increments the run number
    func_image = create_key('sub-{subject}/ses-{session}/func/sub-{subject}_ses-{session}_task-image_run-{item:01d}_bold')

    # Fieldmaps — {item:01d} auto-increments the acquisition number
    fmap = create_key('sub-{subject}/ses-{session}/fmap/sub-{subject}_ses-{session}_acq-{item:01d}_epi')

    info = {t1w: [], t2w: [], func_image: [], fmap: []}
```

**2. MATCHES** — conditional logic that assigns each scan series to a key:
```python
    for s in seqinfo:
        # T1w: MPRAGE with expected dimensions
        if (s.dim1 == 256) and (s.dim2 == 192) and ('MPRAGE_TI1100_ipat2' in s.series_id):
            info[t1w].append(s.series_id)

        # T2w: SPACE with expected dimensions
        if (s.dim1 == 256) and (s.dim3 == 176) and ('T2_1mm_SPACE' in s.series_id):
            info[t2w].append(s.series_id)

        # BOLD: full runs only (395 volumes), filter out any partial scans
        if (s.dim4 == 395) and ('BOLD_IMAGE' in s.series_id):
            info[func_image].append(s.series_id)

        # Fieldmaps
        if 'FieldMap_PA' in s.series_id:
            info[fmap].append(s.series_id)

    return info
```

> **Geoscan note:** The geoscan project had 5 BOLD runs of 395 volumes each (84×84×56 voxels), a T1w MPRAGE (256×192×160), a T2w SPACE (256×256×176), and PA-direction EPI fieldmaps. Sessions were labeled `t2` and `t3`.

Once `heuristic.py` is ready, **delete the `.heudiconv` folder** before running Phase 2:

In [ ]:
# Delete the Phase 1 heuristic output so Phase 2 runs cleanly
# Only run this after you have written and saved heuristic.py
!rm -fr heudiconv/.heudiconv

---
## Phase 2: Convert DICOMs to BIDS NIfTIs

This phase performs the actual DICOM-to-NIfTI conversion using `dcm2niix` (called via heudiconv) and places all output files in the correct BIDS directory structure according to your `heuristic.py`.

Run this single-subject version first to verify the output before running the batch notebook.

In [ ]:
# Confirm heuristic.py is in place before proceeding
if not os.path.exists(heuristic):
    raise FileNotFoundError(f"heuristic.py not found at: {heuristic}\nComplete Phase 1 first.")
print(f"heuristic.py found: {heuristic}")

In [ ]:
# Phase 2: Convert DICOMs to BIDS NIfTIs
#
# -f heuristic.py : use your mapping file for BIDS naming
# -c dcm2niix     : convert using dcm2niix
# -b              : generate BIDS sidecar JSON files
# --overwrite     : overwrite any existing output for this subject

!singularity run --cleanenv \
    -B {project_dir}:/base \
    -B /fmriDataRaw/fmri_data_raw:/raw \
    {heudiconv_sif} \
    -d {raw_dicom_pattern} \
    -o /base/data/bids_data/ \
    -f /base/scripts/BIDS/heudiconv/code/heuristic.py \
    -s {sub} \
    -ss {session} \
    -c dcm2niix -b \
    --overwrite

### Verify the BIDS output

In [ ]:
# List the output files for the converted subject
sub_bids_dir = os.path.join(bids_dir, f'sub-{sub}', f'ses-{session}')

if os.path.exists(sub_bids_dir):
    for root, dirs, files in os.walk(sub_bids_dir):
        rel = os.path.relpath(root, bids_dir)
        print(f'\n{rel}/')
        for f in sorted(files):
            print(f'  {f}')
else:
    print(f"Output directory not found: {sub_bids_dir}")
    print("Conversion may have failed — check the output above for errors.")

---
## Next Steps

1. Verify the BIDS output looks correct (correct scan types, no missing files)
2. Run `heudiconv_batch.ipynb` to convert all remaining subjects
3. Run `fieldmap_intendedfor.ipynb` to add `IntendedFor` fields to fieldmap JSON sidecars
4. Validate the full dataset with the [BIDS Validator](https://bids-standard.github.io/bids-validator/)